In [ ]:
# 1. Imports
import numpy as np
import pandas as pd

from src.likelihoods.planck_shoes_joint import PlanckSH0ESJointLikelihood
from src.likelihoods.desi_bao import DESIBAO
from src.likelihoods.cosmic_chronometers import CosmicChronometers
from src.likelihoods.joint_likelihood import JointLikelihood
from src.physics.mcmc_joint_pipeline import JointMCMCPipeline
from src.pipeline.paper_figures_pipeline import PaperFiguresPipeline

# 2. Load data
planck = "data/planck/planck_distance_priors.csv"
bao = "data/desi/desi_bao.csv"
cc = "data/cc/cosmic_chronometers.csv"

planck_shoes = PlanckSH0ESJointLikelihood(planck)
desi_bao = DESIBAO(bao)
cc_like = CosmicChronometers(cc)

joint = JointLikelihood(planck_shoes, desi_bao, cc_like)

# 3. Define S.T.A.R. Model
H_expr = "H0*sqrt(Ωm*(1+z)**3 + ΩΛ + a*z + b*z**2)"
param_names = ["H0", "Ωm", "ΩΛ", "a", "b"]

priors = {
    "H0": (70, 5),
    "Ωm": (0.3, 0.1),
    "ΩΛ": (0.7, 0.1),
    "a": (0, 0.1),
    "b": (0, 0.1),
}

proposal_widths = {k: 0.01 for k in param_names}

# 4. Run MCMC
pipeline = JointMCMCPipeline(H_expr, param_names, priors, proposal_widths, joint)
chain = pipeline.run(theta0=[70, 0.3, 0.7, 0, 0], nsteps=8000)

# 5. Generate paper figures
fig_pipeline = PaperFiguresPipeline(
    chain,
    param_names,
    H_expr,
    data_paths={"planck": planck, "bao": bao, "cc": cc}
)

constraints = fig_pipeline.run("paper_figures")

constraints
